# Fan out real-time order events from Kinesis Data Streams into Redshift via Lambda

The payments API now requires sub-minute freshness in the reporting warehouse. Reporting still polls Redshift hourly. The CTO wants the dashboard to reflect orders within a minute. You have one session to wire a 1-shard Kinesis Data Stream to a Lambda consumer that batches 100 records and uses the Redshift Data API to insert rows into a staging table, then triggers a stored procedure to merge the staging rows into the fact table. Prove the 45-second SLA against a live event burst.

The literal scenario from the cert YAML is a real-time order event stream. This lab uses a deterministic burst of 500 records across 5 batches so cluster create plus end-to-end latency proof fits inside the 70-minute session window. The exam pattern (Kinesis to Lambda to Redshift Data API; staging plus MERGE; latency proof) holds at both scales; only volume changes.

The four deliverables map to four checkpoints:

1. A single-node `dc2.large` Redshift cluster, a 1-shard Kinesis Data Stream, and a 2-hour safety-net Lambda plus EventBridge rule. All four critical resources are registered in `CLEANUP_MANIFEST` with `critical=True` before any waiter polls.
2. A Lambda consumer wired to the stream via event-source mapping with `BatchSize=100`, `StartingPosition=LATEST`, `State=Enabled`, and `Runtime=python3.12` on the consumer role.
3. A 500-record burst across 5 batches. CloudWatch Logs Insights shows at least 5 Lambda invocations with `recordCount >= 100`, and the Redshift staging table `order_events_stage` has at least 500 rows after a 60-second wait.
4. End-to-end latency from `put_records` to fact-table visibility under 45 seconds. Records carry a `produced_at` epoch-millisecond field; the stored procedure copies it into `orders_fact`; the checkpoint queries `SELECT max(produced_at) FROM orders_fact` and asserts `now - produced_at < 45000` ms, plus all 500 rows are present.

**Time.** About 70 minutes hands-on. The cluster waiter (4 to 6 minutes) and the cleanup waiter (similar) dominate wall-clock; everything else is short.

**Cost.** SECOND HOURLY METER WARNING. Two hourly-billed resources this time: Redshift `dc2.large` at $0.25 per hour ($6 per day; $42 per week if forgotten) and a Kinesis Data Stream at $0.015 per shard-hour for 1 shard ($0.36 per day; $2.52 per week). Idle shards still bill. The cleanup cell tears down Redshift first (higher rate), then the stream, then Lambda. A typical session is well under an hour, so the happy-path cost is $0.20 to $0.30. The 2-hour EventBridge plus Lambda safety net guarantees the cluster, the stream, the safety-net Lambda, and the EventBridge rule auto-destroy at the 2-hour wall-clock mark even if your kernel dies; worst case is $0.60 per session. Three independent failures (cleanup skipped, atexit fails, safety-net Lambda fails) would have to compound to leave the resources running overnight. Set the $20 per month billing alert in your AWS console before you start.

This lab maps to AWS DEA-C01 Domain 1 (Data Ingestion and Transformation) and Domain 2 (Data Store Management); the two combined are 60% of exam weight.


In [ ]:
# NBVAL_SKIP
# Install labrun-checks. Pinned to a specific version per LAB-CREATION-BLUEPRINT
# build rules. Never use unpinned installs.

!pip install --quiet labrun-checks==0.3.0


In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants. Resource names follow the
# labrun-{lab-slug}-{descriptor} pattern from LAB-CREATION-BLUEPRINT
# section 12. Lab 08 is the second critical-resource lab in the DEA-C01
# sequence; the safety-net Lambda plus EventBridge schedule rule are
# declared here so the cluster and the stream cannot outlive the 2-hour
# wall-clock window. Two meters this time: Redshift at $0.25 per hour,
# Kinesis at $0.015 per shard-hour for 1 shard.

import atexit
import getpass
import io
import json
import random
import string
import sys
import time
import urllib.request
import zipfile
from datetime import datetime, timedelta, timezone

import boto3
from botocore.exceptions import ClientError

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CleanupResource,
    CheckpointResult,
)

LAB_ID = "kinesis-streams-lambda-redshift"
LAB_TAG_KEY = "labrun:lab-slug"
LAB_TAG_VALUE = LAB_ID  # must equal cert YAML labs[7].slug exactly
REGION = "us-east-1"  # all DEA-C01 labs run in us-east-1 per LAB-CREATION-BLUEPRINT section 15

# Resource names. Bucket name appends the account ID for global uniqueness.
CLUSTER_ID = f"labrun-{LAB_ID}"  # 39 chars; Redshift cluster identifier max is 63
STREAM_NAME = f"labrun-{LAB_ID}-stream"
CONSUMER_ROLE_NAME = f"labrun-{LAB_ID}-consumer-role"
CONSUMER_LAMBDA_NAME = f"labrun-{LAB_ID}-consumer"
COPY_ROLE_NAME = f"labrun-{LAB_ID}-copy-role"
SG_NAME = f"labrun-{LAB_ID}-sg"
SAFETY_NET_LAMBDA_NAME = f"labrun-{LAB_ID}-safety-net"  # 49 chars; Lambda max 64
SAFETY_NET_RULE_NAME = f"labrun-{LAB_ID}-safety-net-rule"  # 54 chars; EB max 64
SAFETY_NET_ROLE_NAME = f"labrun-{LAB_ID}-safety-net-role"
BUCKET_NAME = None  # resolved after STS once the account ID is known
EVENT_MAPPING_UUID = None  # set in Task 2 once create_event_source_mapping returns

# Cluster shape. dc2.large matches Lab 06; cheapest provisioned node type.
NODE_TYPE = "dc2.large"
NUMBER_OF_NODES = 1
DB_NAME = "labrun"
DB_USER = "labrun_admin"
DB_PORT = 5439

# Stream shape. 1 shard, never more, per RESOURCE-SAFETY-SPEC Section 3 Lab 8.
SHARD_COUNT = 1

# Generated admin password. Redshift requires uppercase, lowercase, and a digit.
_pw_chars = string.ascii_letters + string.digits
_rng = random.Random(0xF00DBEEF)
DB_PASSWORD = "Lab" + "".join(_rng.choice(_pw_chars) for _ in range(20)) + "9"

# Producer burst shape. Deterministic so row counts are stable across runs.
SEED = 42
BURST_BATCHES = 5
BURST_RECORDS_PER_BATCH = 100
BURST_TOTAL_RECORDS = BURST_BATCHES * BURST_RECORDS_PER_BATCH  # 500

# Latency SLA for Checkpoint 4 (milliseconds).
LATENCY_SLA_MS = 45_000

# Safety-net schedule.
SAFETY_NET_HOURS = 2


In [ ]:
# NBVAL_SKIP
# Register the session, validate AWS credentials via STS GetCallerIdentity,
# and confirm the region. This cell must succeed before the manifest cell
# creates anything per LAB-CREATION-BLUEPRINT section 15.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
aws_access_key_id = getpass.getpass("AWS_ACCESS_KEY_ID: ")
aws_secret_access_key = getpass.getpass("AWS_SECRET_ACCESS_KEY: ")
aws_session_token = getpass.getpass(
    "AWS_SESSION_TOKEN (leave blank for long-lived credentials): "
).strip()

AWS_CREDENTIALS = {
    "aws_access_key_id": aws_access_key_id,
    "aws_secret_access_key": aws_secret_access_key,
    "region": REGION,
}
if aws_session_token:
    AWS_CREDENTIALS["aws_session_token"] = aws_session_token

sts = boto3.client(
    "sts",
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    aws_session_token=aws_session_token or None,
    region_name=REGION,
)
try:
    identity = sts.get_caller_identity()
except ClientError as e:
    print("Credentials are missing or expired. STS GetCallerIdentity failed:")
    print(f"  {e}")
    print()
    print("Refresh your AWS credentials and re-run this cell.")
    raise SystemExit(1)

ACCOUNT_ID = identity["Account"]
CALLER_ARN = identity["Arn"]
print(f"Credentials validated. Account: {ACCOUNT_ID}")
print(f"Caller ARN: {CALLER_ARN}")
print(f"Region: {REGION}")
print(f"Session token in use: {bool(aws_session_token)}")

BUCKET_NAME = f"labrun-{LAB_ID}-{ACCOUNT_ID}"
print(f"Bucket name resolved: {BUCKET_NAME}")

# Register the session with labrun-checks. register() returns None;
# do not assign its return value.
register(session_token=session_token, lab_id=LAB_ID, credentials=AWS_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")


In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, and orphan scan.
#
# Per RESOURCE-SAFETY-SPEC Rule 2, the four hourly-billed (or safety-net)
# resources tear down FIRST: cluster (highest rate), stream (idle shards
# still bill), safety-net EventBridge rule, then safety-net Lambda. The
# event-source mapping comes after to unwire the consumer cleanly; then
# the standard resources (consumer Lambda, IAM roles, security group,
# COPY role, S3 bucket). The manifest below lists every resource the lab
# creates in strict reverse-creation, critical-first order so the atexit
# handler tears down the meters even if the kernel dies during the
# create_cluster waiter or the create_stream waiter.
#
# Per RESOURCE-SAFETY-SPEC Rule 4, the orphan scan blocks execution if
# any tagged resources from a prior session exist (not just print a
# warning).
#
# labrun-checks 0.3.0 lacks adapters for redshift_cluster, kinesis_stream,
# eventbridge_rule, lambda_function, lambda_event_source_mapping, and
# security_group. The cleanup cell at the bottom of this notebook defines
# an inline _LabAdapter that wraps AwsCleanupAdapter and dispatches those
# six types inline. The new resource types are still declared declaratively
# here so the canonical summary, atexit handler, and orphan scan all see
# them.

CLEANUP_MANIFEST = [
    CleanupResource(
        type="redshift_cluster",
        id=CLUSTER_ID,
        region=REGION,
        cli_delete_command=(
            f"aws redshift delete-cluster --cluster-identifier {CLUSTER_ID} "
            f"--skip-final-cluster-snapshot"
        ),
        critical=True,
    ),
    CleanupResource(
        type="kinesis_stream",
        id=STREAM_NAME,
        region=REGION,
        cli_delete_command=(
            f"aws kinesis delete-stream --stream-name {STREAM_NAME} "
            f"--enforce-consumer-deletion"
        ),
        critical=True,
    ),
    CleanupResource(
        type="eventbridge_rule",
        id=SAFETY_NET_RULE_NAME,
        region=REGION,
        cli_delete_command=(
            f"aws events remove-targets --rule {SAFETY_NET_RULE_NAME} --ids 1 && "
            f"aws events delete-rule --name {SAFETY_NET_RULE_NAME}"
        ),
        critical=True,
    ),
    CleanupResource(
        type="lambda_function",
        id=SAFETY_NET_LAMBDA_NAME,
        region=REGION,
        cli_delete_command=(
            f"aws lambda delete-function --function-name {SAFETY_NET_LAMBDA_NAME}"
        ),
        critical=True,
    ),
    CleanupResource(
        type="lambda_event_source_mapping",
        id="(set after create_event_source_mapping)",
        region=REGION,
        cli_delete_command=(
            "aws lambda delete-event-source-mapping --uuid <uuid set in Task 2>"
        ),
    ),
    CleanupResource(
        type="lambda_function",
        id=CONSUMER_LAMBDA_NAME,
        region=REGION,
        cli_delete_command=(
            f"aws lambda delete-function --function-name {CONSUMER_LAMBDA_NAME}"
        ),
    ),
    CleanupResource(
        type="iam_role",
        id=CONSUMER_ROLE_NAME,
        region=REGION,
        cli_delete_command=f"aws iam delete-role --role-name {CONSUMER_ROLE_NAME}",
    ),
    CleanupResource(
        type="iam_role",
        id=SAFETY_NET_ROLE_NAME,
        region=REGION,
        cli_delete_command=f"aws iam delete-role --role-name {SAFETY_NET_ROLE_NAME}",
    ),
    CleanupResource(
        type="security_group",
        id=SG_NAME,
        region=REGION,
        cli_delete_command=f"aws ec2 delete-security-group --group-name {SG_NAME}",
    ),
    CleanupResource(
        type="iam_role",
        id=COPY_ROLE_NAME,
        region=REGION,
        cli_delete_command=f"aws iam delete-role --role-name {COPY_ROLE_NAME}",
    ),
    CleanupResource(
        type="s3_bucket",
        id=BUCKET_NAME,
        region=REGION,
        cli_delete_command=f"aws s3 rb s3://{BUCKET_NAME} --force",
    ),
]


def _atexit_cleanup() -> None:
    """Best-effort teardown on kernel shutdown.

    The dedicated cleanup cell is the authoritative entry point; this is
    the safety net for kernel crashes during the create_cluster waiter,
    the create_stream waiter, or any other long-running step. Errors are
    swallowed because atexit handlers must not raise.
    """
    try:
        run_cleanup(CLEANUP_MANIFEST, adapter=_LabAdapter()) if "_LabAdapter" in globals() else run_cleanup(CLEANUP_MANIFEST)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans() -> list[str]:
    """Refuse to start if a previous run left tagged resources behind.

    Per RESOURCE-SAFETY-SPEC Rule 4, the setup cell must check for leftover
    state with the lab's tag and surface the cleanup command before
    creating any new resources. Critical for Lab 08 because a leftover
    cluster is $0.25 per hour and a leftover shard is $0.015 per hour.
    """
    client = boto3.client(
        "resourcegroupstaggingapi",
        aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
        aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
        aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
        region_name=REGION,
    )
    arns: list[str] = []
    paginator = client.get_paginator("get_resources")
    for page in paginator.paginate(
        TagFilters=[{"Key": LAB_TAG_KEY, "Values": [LAB_TAG_VALUE]}],
    ):
        for item in page.get("ResourceTagMappingList", []):
            arns.append(item.get("ResourceARN", ""))
    return arns


orphans = scan_for_orphans()
if orphans:
    print(f"BLOCKED: existing resources tagged labrun:lab-slug={LAB_TAG_VALUE} were found:")
    for arn in orphans:
        print("  -", arn)
    print()
    print("These are leftovers from a previous run of this lab. Re-running")
    print("setup against an unclean state can produce duplicate resources")
    print("or, worse, a second Redshift cluster and a second Kinesis shard")
    print("billing in parallel. Run the cleanup cell at the bottom of this")
    print("notebook first, or remove them manually with the AWS CLI commands")
    print("the cleanup cell prints, then re-run setup.")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")


## Task 1: Provision the Redshift cluster, the Kinesis stream, and the safety-net Lambda; register all four as critical before any waiter polls

This is the hottest cell of the lab. Two meters start in this cell: the Redshift `dc2.large` node ($0.25 per hour) and the Kinesis 1-shard stream ($0.015 per shard-hour). The order of operations matters. Critical resources go into `CLEANUP_MANIFEST` (already pre-declared at module level) and the safety-net Lambda is wired BEFORE the cluster and stream waiters start polling. If the kernel dies during a waiter, the atexit handler still has the manifest entries and can delete both meters.

Eight steps in this cell, in order:

1. **Create the S3 bucket and the COPY role.** The bucket holds the Lambda deployment zip and any audit logs. The COPY role is the IAM role Redshift assumes to read from S3 (kept for symmetry with Lab 06; the Data API path used by the Lambda consumer does not require it for staging-table inserts, but it is needed for any COPY-based fallback).
2. **Create the consumer IAM role.** Trust policy: `lambda.amazonaws.com`. Inline policy: `kinesis:GetRecords`, `kinesis:GetShardIterator`, `kinesis:DescribeStream`, `kinesis:ListShards` on the stream ARN, plus `redshift-data:ExecuteStatement`, `redshift-data:DescribeStatement`, `redshift-data:GetStatementResult` on `*`, plus `logs:CreateLogGroup`, `logs:CreateLogStream`, `logs:PutLogEvents` on `*`. The logs statement is non-negotiable; without it the Lambda appears not to run because logs never appear.
3. **Create the security group locked to your current public IP.** Port 5439 inbound, default VPC. The IP comes from `https://checkip.amazonaws.com` (an AWS-owned endpoint; no third-party fetch).
4. **Create the safety-net IAM role and Lambda.** The Lambda has one job: at the 2-hour wall-clock mark, tag-check then delete the cluster, the stream, the safety-net Lambda itself, the EventBridge rule, and the safety-net role. Tag-scoped: each delete confirms `labrun:lab-slug == kinesis-streams-lambda-redshift` first so a misconfigured safety net cannot delete an unrelated resource.
5. **Create the EventBridge schedule rule and wire it to the safety-net Lambda.** One-shot cron at `now + 2h`. `add_permission` lets `events.amazonaws.com` invoke the function; `put_targets` ties the rule to the Lambda ARN.
6. **Create the Kinesis Data Stream.** Exactly 1 shard. `CreateStream` returns immediately; the stream-active waiter takes ~30 seconds. Tag at creation so the safety net's tag check matches.
7. **Create the Redshift cluster.** Single-node `dc2.large`, `PubliclyAccessible=True`, the lab SG attached, the COPY role attached via `IamRoles`. `create_cluster` returns in seconds; the cluster-available waiter takes 4 to 6 minutes.
8. **Create the staging table, the fact table, and the stored procedure.** Use the Redshift Data API. `order_events_stage` holds incoming records before the MERGE; `orders_fact` is the read-side table; `merge_order_events()` copies staging rows into fact and truncates staging.

Checkpoint 1 confirms the cluster is `available`, the stream is `ACTIVE` with `OpenShardCount == 1`, the EventBridge rule and safety-net Lambda exist, and `CLEANUP_MANIFEST` carries `critical=True` on all four (cluster, stream, EB rule, safety-net Lambda).


In [ ]:
# NBVAL_SKIP
# Task 1: Bucket, COPY role, consumer role, security group, safety-net
# Lambda plus EventBridge rule, Kinesis stream, Redshift cluster, then the
# staging plus fact tables and the merge stored procedure. The order below
# is deliberate; the cluster and the stream are created LAST so the safety
# net is already in place when both meters start.

s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
iam = boto3.client(
    "iam",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
ec2 = boto3.client(
    "ec2",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
redshift = boto3.client(
    "redshift",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
kinesis = boto3.client(
    "kinesis",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
events = boto3.client(
    "events",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
lambda_client = boto3.client(
    "lambda",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
redshift_data = boto3.client(
    "redshift-data",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# 1. Bucket. us-east-1 rejects LocationConstraint; other regions require it.
# YOUR CODE: s3.create_bucket(Bucket=BUCKET_NAME).
s3.put_bucket_tagging(
    Bucket=BUCKET_NAME,
    Tagging={"TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]},
)
print(f"Bucket created and tagged: {BUCKET_NAME}")

# 2. COPY role (Redshift service role for any S3 reads).
copy_trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "redshift.amazonaws.com"},
        "Action": "sts:AssumeRole",
    }],
}
copy_inline_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Action": ["s3:GetObject", "s3:ListBucket"],
        "Resource": [
            f"arn:aws:s3:::{BUCKET_NAME}",
            f"arn:aws:s3:::{BUCKET_NAME}/*",
        ],
    }],
}
# YOUR CODE: iam.create_role(
#   RoleName=COPY_ROLE_NAME,
#   AssumeRolePolicyDocument=json.dumps(copy_trust_policy),
#   Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
# ).
# YOUR CODE: iam.put_role_policy(
#   RoleName=COPY_ROLE_NAME,
#   PolicyName="labrun-copy-policy",
#   PolicyDocument=json.dumps(copy_inline_policy),
# ).
copy_role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{COPY_ROLE_NAME}"
print(f"COPY role ARN: {copy_role_arn}")

# 3. Consumer role (Lambda execution role for the Kinesis consumer).
consumer_trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "lambda.amazonaws.com"},
        "Action": "sts:AssumeRole",
    }],
}
stream_arn = f"arn:aws:kinesis:{REGION}:{ACCOUNT_ID}:stream/{STREAM_NAME}"
consumer_inline_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "kinesis:GetRecords",
                "kinesis:GetShardIterator",
                "kinesis:DescribeStream",
                "kinesis:DescribeStreamSummary",
                "kinesis:ListShards",
                "kinesis:SubscribeToShard",
            ],
            "Resource": stream_arn,
        },
        {
            "Effect": "Allow",
            "Action": [
                "redshift-data:ExecuteStatement",
                "redshift-data:DescribeStatement",
                "redshift-data:GetStatementResult",
            ],
            "Resource": "*",
        },
        {
            "Effect": "Allow",
            "Action": [
                "redshift:GetClusterCredentials",
            ],
            "Resource": "*",
        },
        {
            "Effect": "Allow",
            "Action": [
                "logs:CreateLogGroup",
                "logs:CreateLogStream",
                "logs:PutLogEvents",
            ],
            "Resource": "*",
        },
    ],
}
# YOUR CODE: iam.create_role(
#   RoleName=CONSUMER_ROLE_NAME,
#   AssumeRolePolicyDocument=json.dumps(consumer_trust_policy),
#   Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
# ).
# YOUR CODE: iam.put_role_policy(
#   RoleName=CONSUMER_ROLE_NAME,
#   PolicyName="labrun-consumer-policy",
#   PolicyDocument=json.dumps(consumer_inline_policy),
# ).
consumer_role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{CONSUMER_ROLE_NAME}"
print(f"Consumer role ARN: {consumer_role_arn}")

# 4. Security group locked to the caller's public IP. Port 5439 inbound.
my_ip = urllib.request.urlopen("https://checkip.amazonaws.com").read().decode().strip()
my_cidr = f"{my_ip}/32"
default_vpc = ec2.describe_vpcs(Filters=[{"Name": "is-default", "Values": ["true"]}])
default_vpc_id = default_vpc["Vpcs"][0]["VpcId"]
sg_resp = ec2.create_security_group(
    GroupName=SG_NAME,
    Description=f"Redshift access for {LAB_ID}, locked to caller IP",
    VpcId=default_vpc_id,
    TagSpecifications=[{
        "ResourceType": "security-group",
        "Tags": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
    }],
)
sg_id = sg_resp["GroupId"]
ec2.authorize_security_group_ingress(
    GroupId=sg_id,
    IpPermissions=[{
        "IpProtocol": "tcp",
        "FromPort": DB_PORT,
        "ToPort": DB_PORT,
        "IpRanges": [{"CidrIp": my_cidr, "Description": "labrun lab caller IP"}],
    }],
)
print(f"Security group created: {sg_id} ({SG_NAME}); ingress 5439 from {my_cidr}")

# 5. Safety-net IAM role + Lambda + EventBridge rule.
sn_trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "lambda.amazonaws.com"},
        "Action": "sts:AssumeRole",
    }],
}
sn_inline_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "redshift:DescribeClusters",
                "redshift:DeleteCluster",
                "redshift:DescribeTags",
                "kinesis:DescribeStream",
                "kinesis:DescribeStreamSummary",
                "kinesis:DeleteStream",
                "kinesis:ListTagsForStream",
                "lambda:GetFunction",
                "lambda:DeleteFunction",
                "lambda:ListTags",
                "events:DescribeRule",
                "events:DeleteRule",
                "events:RemoveTargets",
                "events:ListTagsForResource",
                "iam:DeleteRole",
                "iam:DeleteRolePolicy",
                "iam:ListRolePolicies",
                "iam:GetRole",
            ],
            "Resource": "*",
        },
        {
            "Effect": "Allow",
            "Action": [
                "logs:CreateLogGroup",
                "logs:CreateLogStream",
                "logs:PutLogEvents",
            ],
            "Resource": "*",
        },
    ],
}
# YOUR CODE: iam.create_role(
#   RoleName=SAFETY_NET_ROLE_NAME,
#   AssumeRolePolicyDocument=json.dumps(sn_trust_policy),
#   Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
# ).
# YOUR CODE: iam.put_role_policy(
#   RoleName=SAFETY_NET_ROLE_NAME,
#   PolicyName="labrun-safety-net-policy",
#   PolicyDocument=json.dumps(sn_inline_policy),
# ).
sn_role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{SAFETY_NET_ROLE_NAME}"

# Safety-net Lambda source. Tag-scoped: each delete confirms
# labrun:lab-slug matches this lab before calling delete. boto3.client
# inside this source string uses the Lambda execution role and does not
# need aws_session_token (exception noted; the notebook-level rule does
# require session token on every boto3.client call).
lambda_source = f"""
import boto3

CLUSTER_ID = "{CLUSTER_ID}"
STREAM_NAME = "{STREAM_NAME}"
SAFETY_NET_LAMBDA = "{SAFETY_NET_LAMBDA_NAME}"
SAFETY_NET_RULE = "{SAFETY_NET_RULE_NAME}"
SAFETY_NET_ROLE = "{SAFETY_NET_ROLE_NAME}"
EXPECTED_TAG_KEY = "{LAB_TAG_KEY}"
EXPECTED_TAG_VALUE = "{LAB_TAG_VALUE}"


def _matches(tags):
    return tags.get(EXPECTED_TAG_KEY) == EXPECTED_TAG_VALUE


def handler(event, context):
    # boto3.client below uses the Lambda execution role; no session token.
    rs = boto3.client("redshift")
    kin = boto3.client("kinesis")
    lam = boto3.client("lambda")
    eb = boto3.client("events")
    iam = boto3.client("iam")
    results = {{}}

    try:
        resp = rs.describe_clusters(ClusterIdentifier=CLUSTER_ID)
        cluster = resp["Clusters"][0]
        tags = {{t["Key"]: t["Value"] for t in cluster.get("Tags", [])}}
        if _matches(tags):
            rs.delete_cluster(
                ClusterIdentifier=CLUSTER_ID,
                SkipFinalClusterSnapshot=True,
            )
            results["cluster"] = "deleted"
        else:
            results["cluster"] = "refused (tag mismatch)"
    except rs.exceptions.ClusterNotFoundFault:
        results["cluster"] = "absent"
    except Exception as exc:
        results["cluster"] = f"error: {{exc}}"

    try:
        kin.describe_stream(StreamName=STREAM_NAME)
        tag_resp = kin.list_tags_for_stream(StreamName=STREAM_NAME)
        tags = {{t["Key"]: t["Value"] for t in tag_resp.get("Tags", [])}}
        if _matches(tags):
            kin.delete_stream(
                StreamName=STREAM_NAME,
                EnforceConsumerDeletion=True,
            )
            results["stream"] = "deleted"
        else:
            results["stream"] = "refused (tag mismatch)"
    except kin.exceptions.ResourceNotFoundException:
        results["stream"] = "absent"
    except Exception as exc:
        results["stream"] = f"error: {{exc}}"

    try:
        eb.remove_targets(Rule=SAFETY_NET_RULE, Ids=["1"], Force=True)
    except Exception:
        pass
    try:
        eb.delete_rule(Name=SAFETY_NET_RULE)
        results["rule"] = "deleted"
    except Exception as exc:
        results["rule"] = f"error: {{exc}}"

    try:
        for pn in iam.list_role_policies(RoleName=SAFETY_NET_ROLE).get("PolicyNames", []):
            iam.delete_role_policy(RoleName=SAFETY_NET_ROLE, PolicyName=pn)
        iam.delete_role(RoleName=SAFETY_NET_ROLE)
        results["role"] = "deleted"
    except Exception as exc:
        results["role"] = f"error: {{exc}}"

    try:
        lam.delete_function(FunctionName=SAFETY_NET_LAMBDA)
        results["self"] = "deleted"
    except Exception as exc:
        results["self"] = f"error: {{exc}}"

    print(results)
    return results
"""

zbuf = io.BytesIO()
with zipfile.ZipFile(zbuf, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.writestr("index.py", lambda_source)
zip_bytes = zbuf.getvalue()

# IAM role propagation can take a few seconds before Lambda will accept it.
_lambda_deadline = time.time() + 60
while True:
    try:
        # YOUR CODE: lambda_client.create_function(
        #   FunctionName=SAFETY_NET_LAMBDA_NAME,
        #   Runtime="python3.12",
        #   Role=sn_role_arn,
        #   Handler="index.handler",
        #   Code={"ZipFile": zip_bytes},
        #   Timeout=120,
        #   Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
        # ).
        break
    except ClientError as e:
        if e.response["Error"]["Code"] == "InvalidParameterValueException" and time.time() < _lambda_deadline:
            time.sleep(5)
            continue
        raise
lambda_arn = f"arn:aws:lambda:{REGION}:{ACCOUNT_ID}:function:{SAFETY_NET_LAMBDA_NAME}"
print(f"Safety-net Lambda created: {lambda_arn}")

# EventBridge schedule rule: one-shot at lab_start + SAFETY_NET_HOURS.
fire_at = (datetime.now(timezone.utc) + timedelta(hours=SAFETY_NET_HOURS)).replace(microsecond=0)
cron_expr = (
    f"cron({fire_at.minute} {fire_at.hour} "
    f"{fire_at.day} {fire_at.month} ? {fire_at.year})"
)
# YOUR CODE: events.put_rule(
#   Name=SAFETY_NET_RULE_NAME,
#   ScheduleExpression=cron_expr,
#   State="ENABLED",
#   Description=f"Safety net for {LAB_ID}; auto-deletes cluster + stream at +{SAFETY_NET_HOURS}h.",
#   Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
# ).
# YOUR CODE: lambda_client.add_permission(
#   FunctionName=SAFETY_NET_LAMBDA_NAME,
#   StatementId="labrun-eventbridge-invoke",
#   Action="lambda:InvokeFunction",
#   Principal="events.amazonaws.com",
#   SourceArn=f"arn:aws:events:{REGION}:{ACCOUNT_ID}:rule/{SAFETY_NET_RULE_NAME}",
# ).
# YOUR CODE: events.put_targets(
#   Rule=SAFETY_NET_RULE_NAME,
#   Targets=[{"Id": "1", "Arn": lambda_arn}],
# ).
print(f"EventBridge rule {SAFETY_NET_RULE_NAME} scheduled at {cron_expr}")

# 6. Create the Kinesis stream. Single shard, tag at creation.
# YOUR CODE: kinesis.create_stream(
#   StreamName=STREAM_NAME,
#   ShardCount=SHARD_COUNT,
#   StreamModeDetails={"StreamMode": "PROVISIONED"},
#   Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
# ).
print(f"create_stream returned for {STREAM_NAME}; shard meter started at $0.015/shard-hour.")
stream_waiter = kinesis.get_waiter("stream_exists")
stream_waiter.wait(
    StreamName=STREAM_NAME,
    WaiterConfig={"Delay": 5, "MaxAttempts": 30},
)
print(f"Kinesis stream {STREAM_NAME} is ACTIVE.")

# 7. Create the Redshift cluster.
# YOUR CODE: redshift.create_cluster(
#   ClusterIdentifier=CLUSTER_ID,
#   NodeType=NODE_TYPE,
#   NumberOfNodes=NUMBER_OF_NODES,
#   ClusterType="single-node",
#   MasterUsername=DB_USER,
#   MasterUserPassword=DB_PASSWORD,
#   DBName=DB_NAME,
#   PubliclyAccessible=True,
#   VpcSecurityGroupIds=[sg_id],
#   IamRoles=[copy_role_arn],
#   Port=DB_PORT,
#   Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
# ).
print(f"create_cluster returned for {CLUSTER_ID}; cluster meter started at $0.25/hour.")
print("Waiting for cluster to become available (4 to 6 minutes typical)...")
waiter = redshift.get_waiter("cluster_available")
waiter.wait(
    ClusterIdentifier=CLUSTER_ID,
    WaiterConfig={"Delay": 30, "MaxAttempts": 30},
)
print(f"Cluster {CLUSTER_ID} is available.")

# 8. Create the staging table, the fact table, and the merge stored procedure.

def _run_sql_until_finished(sql: str) -> dict:
    start = redshift_data.execute_statement(
        ClusterIdentifier=CLUSTER_ID,
        Database=DB_NAME,
        DbUser=DB_USER,
        Sql=sql,
    )
    sid = start["Id"]
    deadline = time.time() + 120
    while time.time() < deadline:
        desc = redshift_data.describe_statement(Id=sid)
        status_ = desc["Status"]
        if status_ in ("FINISHED", "FAILED", "ABORTED"):
            if status_ != "FINISHED":
                raise RuntimeError(
                    f"redshift-data {sid} ended {status_}: {desc.get('Error', '(no error)')}"
                )
            return desc
        time.sleep(1)
    raise RuntimeError(f"redshift-data {sid} did not finish within 2 minutes.")


_run_sql_until_finished("DROP TABLE IF EXISTS order_events_stage;")
_run_sql_until_finished("DROP TABLE IF EXISTS orders_fact;")
_run_sql_until_finished(
    "CREATE TABLE order_events_stage ("
    "  order_id BIGINT, customer_id INTEGER, amount_cents INTEGER, "
    "  produced_at BIGINT"
    ");"
)
_run_sql_until_finished(
    "CREATE TABLE orders_fact ("
    "  order_id BIGINT, customer_id INTEGER, amount_cents INTEGER, "
    "  produced_at BIGINT"
    ");"
)
_run_sql_until_finished(
    "CREATE OR REPLACE PROCEDURE merge_order_events() "
    "LANGUAGE plpgsql AS $$ "
    "BEGIN "
    "  INSERT INTO orders_fact (order_id, customer_id, amount_cents, produced_at) "
    "  SELECT s.order_id, s.customer_id, s.amount_cents, s.produced_at "
    "  FROM order_events_stage s "
    "  LEFT JOIN orders_fact f ON s.order_id = f.order_id "
    "  WHERE f.order_id IS NULL; "
    "  TRUNCATE order_events_stage; "
    "END; "
    "$$;"
)
print("order_events_stage, orders_fact, and merge_order_events() created.")


In [ ]:
# NBVAL_SKIP
# Checkpoint 1: cluster available, stream ACTIVE with OpenShardCount=1,
# safety-net EB rule and Lambda exist, and CLEANUP_MANIFEST flags all four
# critical resources with critical=True.


def checkpoint_1(session):
    try:
        rs = boto3.client(
            "redshift",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        kin = boto3.client(
            "kinesis",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        eb = boto3.client(
            "events",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        lam = boto3.client(
            "lambda",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        # Cluster.
        try:
            resp = rs.describe_clusters(ClusterIdentifier=CLUSTER_ID)
        except ClientError as e:
            code_ = e.response["Error"]["Code"]
            if code_ in ("ClusterNotFound", "ClusterNotFoundFault"):
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Redshift cluster {CLUSTER_ID!r} does not exist. "
                        f"Run the Task 1 cell to create it."
                    ),
                )
            return CheckpointResult(status="error", error_reason=str(e))
        clusters = resp.get("Clusters", [])
        if not clusters:
            return CheckpointResult(
                status="fail",
                error_reason=f"describe_clusters returned no clusters for {CLUSTER_ID!r}.",
            )
        cluster = clusters[0]
        status_ = cluster.get("ClusterStatus", "")
        if status_ != "available":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Cluster status is {status_!r}, expected 'available'. "
                    f"Wait for the create_cluster waiter to finish."
                ),
            )

        # Stream.
        try:
            summary = kin.describe_stream_summary(StreamName=STREAM_NAME)
        except ClientError as e:
            if e.response["Error"]["Code"] == "ResourceNotFoundException":
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Kinesis stream {STREAM_NAME!r} does not exist. "
                        f"Run the Task 1 cell to create it."
                    ),
                )
            return CheckpointResult(status="error", error_reason=str(e))
        s_desc = summary.get("StreamDescriptionSummary", {})
        s_status = s_desc.get("StreamStatus", "")
        if s_status != "ACTIVE":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Stream status is {s_status!r}, expected 'ACTIVE'. "
                    f"Wait for the stream_exists waiter to finish."
                ),
            )
        open_shards = s_desc.get("OpenShardCount", 0)
        if open_shards != 1:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"OpenShardCount is {open_shards}, expected 1. "
                    f"Per RESOURCE-SAFETY-SPEC Section 3 Lab 8, this lab uses "
                    f"exactly 1 shard so the meter stays at $0.015/shard-hour."
                ),
            )

        # Safety-net EventBridge rule.
        try:
            eb.describe_rule(Name=SAFETY_NET_RULE_NAME)
        except ClientError as e:
            if e.response["Error"]["Code"] == "ResourceNotFoundException":
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"EventBridge rule {SAFETY_NET_RULE_NAME!r} does not "
                        f"exist. Without it, the cluster and stream cannot "
                        f"auto-destroy on kernel crash."
                    ),
                )
            return CheckpointResult(status="error", error_reason=str(e))

        # Safety-net Lambda.
        try:
            lam.get_function(FunctionName=SAFETY_NET_LAMBDA_NAME)
        except ClientError as e:
            if e.response["Error"]["Code"] == "ResourceNotFoundException":
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Safety-net Lambda {SAFETY_NET_LAMBDA_NAME!r} does "
                        f"not exist. The EventBridge rule has no target to "
                        f"invoke; the safety net is non-functional."
                    ),
                )
            return CheckpointResult(status="error", error_reason=str(e))

        # Manifest inspection: cluster, stream, EB rule, safety-net Lambda
        # all must be present with critical=True.
        manifest = globals().get("CLEANUP_MANIFEST", [])
        required = {
            ("redshift_cluster", CLUSTER_ID): None,
            ("kinesis_stream", STREAM_NAME): None,
            ("eventbridge_rule", SAFETY_NET_RULE_NAME): None,
            ("lambda_function", SAFETY_NET_LAMBDA_NAME): None,
        }
        for r in manifest:
            key = (r.type, r.id)
            if key in required:
                required[key] = r
        missing = [k for k, v in required.items() if v is None]
        if missing:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"CLEANUP_MANIFEST is missing entries for: {missing}. "
                    f"All four critical resources must be declared so atexit "
                    f"can tear them down on kernel crash."
                ),
            )
        for key, r in required.items():
            if not getattr(r, "critical", False):
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"CLEANUP_MANIFEST entry for {key} is missing "
                        f"critical=True. Per RESOURCE-SAFETY-SPEC Rule 2 "
                        f"and Rule 5 (companion panel reads this flag), "
                        f"the four critical entries must carry critical=True."
                    ),
                )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(1, checkpoint_1)


<details><summary>Hint 1 (nudge)</summary>

The checkpoint walks four resources in order: Redshift cluster `available`, Kinesis stream `ACTIVE` with `OpenShardCount == 1`, EventBridge rule exists, safety-net Lambda exists. Then it walks `CLEANUP_MANIFEST` for four entries (`redshift_cluster`, `kinesis_stream`, `eventbridge_rule`, `lambda_function` for the safety net) and confirms each carries `critical=True`. Read the failure reason. The most common miss is forgetting `ShardCount=1` (the boto3 default is sometimes 1 but not guaranteed across SDK versions, so set it explicitly). The second most common miss is creating the stream without tags; the safety net's tag check will refuse to delete an untagged stream.

</details>

<details><summary>Hint 2 (direction)</summary>

Eight API calls in this task, plus three SQL statements at the end. `s3.create_bucket(Bucket=BUCKET_NAME)` (no `LocationConstraint` in us-east-1). `iam.create_role(RoleName=COPY_ROLE_NAME, ...)` plus `iam.put_role_policy(...)` for the COPY role; same for the consumer role with the inline policy that includes the Kinesis actions, the Redshift Data API actions, and the logs actions. `iam.create_role(RoleName=SAFETY_NET_ROLE_NAME, ...)` plus `iam.put_role_policy(...)`. `lambda_client.create_function(FunctionName=SAFETY_NET_LAMBDA_NAME, Runtime="python3.12", Role=sn_role_arn, Handler="index.handler", Code={"ZipFile": zip_bytes}, Timeout=120, Tags={LAB_TAG_KEY: LAB_TAG_VALUE})` inside the retry loop. `events.put_rule` plus `lambda_client.add_permission` plus `events.put_targets` to wire the schedule rule. `kinesis.create_stream(StreamName=STREAM_NAME, ShardCount=1, StreamModeDetails={"StreamMode": "PROVISIONED"}, Tags={LAB_TAG_KEY: LAB_TAG_VALUE})`. `redshift.create_cluster(ClusterIdentifier=CLUSTER_ID, NodeType=NODE_TYPE, NumberOfNodes=NUMBER_OF_NODES, ClusterType="single-node", MasterUsername=DB_USER, MasterUserPassword=DB_PASSWORD, DBName=DB_NAME, PubliclyAccessible=True, VpcSecurityGroupIds=[sg_id], IamRoles=[copy_role_arn], Port=DB_PORT, Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}])`. The three SQL statements use the inline `_run_sql_until_finished` helper.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 1:

```python
s3.create_bucket(Bucket=BUCKET_NAME)

iam.create_role(
    RoleName=COPY_ROLE_NAME,
    AssumeRolePolicyDocument=json.dumps(copy_trust_policy),
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
iam.put_role_policy(
    RoleName=COPY_ROLE_NAME,
    PolicyName="labrun-copy-policy",
    PolicyDocument=json.dumps(copy_inline_policy),
)

iam.create_role(
    RoleName=CONSUMER_ROLE_NAME,
    AssumeRolePolicyDocument=json.dumps(consumer_trust_policy),
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
iam.put_role_policy(
    RoleName=CONSUMER_ROLE_NAME,
    PolicyName="labrun-consumer-policy",
    PolicyDocument=json.dumps(consumer_inline_policy),
)

iam.create_role(
    RoleName=SAFETY_NET_ROLE_NAME,
    AssumeRolePolicyDocument=json.dumps(sn_trust_policy),
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
iam.put_role_policy(
    RoleName=SAFETY_NET_ROLE_NAME,
    PolicyName="labrun-safety-net-policy",
    PolicyDocument=json.dumps(sn_inline_policy),
)

# Inside the retry loop:
lambda_client.create_function(
    FunctionName=SAFETY_NET_LAMBDA_NAME,
    Runtime="python3.12",
    Role=sn_role_arn,
    Handler="index.handler",
    Code={"ZipFile": zip_bytes},
    Timeout=120,
    Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
)

events.put_rule(
    Name=SAFETY_NET_RULE_NAME,
    ScheduleExpression=cron_expr,
    State="ENABLED",
    Description=f"Safety net for {LAB_ID}; auto-deletes cluster + stream at +{SAFETY_NET_HOURS}h.",
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
lambda_client.add_permission(
    FunctionName=SAFETY_NET_LAMBDA_NAME,
    StatementId="labrun-eventbridge-invoke",
    Action="lambda:InvokeFunction",
    Principal="events.amazonaws.com",
    SourceArn=f"arn:aws:events:{REGION}:{ACCOUNT_ID}:rule/{SAFETY_NET_RULE_NAME}",
)
events.put_targets(
    Rule=SAFETY_NET_RULE_NAME,
    Targets=[{"Id": "1", "Arn": lambda_arn}],
)

kinesis.create_stream(
    StreamName=STREAM_NAME,
    ShardCount=SHARD_COUNT,
    StreamModeDetails={"StreamMode": "PROVISIONED"},
    Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
)

redshift.create_cluster(
    ClusterIdentifier=CLUSTER_ID,
    NodeType=NODE_TYPE,
    NumberOfNodes=NUMBER_OF_NODES,
    ClusterType="single-node",
    MasterUsername=DB_USER,
    MasterUserPassword=DB_PASSWORD,
    DBName=DB_NAME,
    PubliclyAccessible=True,
    VpcSecurityGroupIds=[sg_id],
    IamRoles=[copy_role_arn],
    Port=DB_PORT,
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
```

If any `create_*` call returns `AccessDenied`, your `labrun-test` IAM user is missing a managed policy from the lab page (`AmazonKinesisFullAccess`, `AWSLambda_FullAccess`, `AmazonRedshiftFullAccess`, `AmazonRedshiftDataFullAccess`, `IAMFullAccess`, `AmazonEC2FullAccess`, `AmazonEventBridgeFullAccess`, `CloudWatchLogsFullAccess`). If `create_stream` returns `LimitExceededException`, your account already has a stream by that name; run the orphan-scan cleanup. If `create_cluster` returns `ClusterAlreadyExists`, same fix; clean the orphan first.

</details>


**Wallet check.** **Two meters started.** Redshift `dc2.large` is now billing at $0.25 per hour. The Kinesis stream is now billing at $0.015 per shard-hour for 1 shard. Four to six minutes for the cluster waiter is about 2 to 3 cents combined ($0.265 per hour blended); the stream-active waiter is ~30 seconds. The S3 bucket, the IAM roles, the security group, the Lambda function, and the EventBridge rule are all free at this volume. The 2-hour safety net caps worst case at $0.60 (cluster + stream + a small Lambda burn). Keep moving; Tasks 2 through 4 are bytes-cheap but the meters run until cleanup deletes them.


## Task 2: Package the consumer Lambda, deploy it, and wire it to the stream via event-source mapping

The Lambda consumer reads records from the Kinesis stream, base64-decodes the payload, batches the rows into a single `INSERT` against `order_events_stage` via the Redshift Data API, then triggers `CALL merge_order_events()` to MERGE the staging rows into `orders_fact`. The function code lives in a string variable in this cell; you zip it in memory and upload it via `lambda.create_function`.

Three pieces in this task:

1. **Package and deploy the consumer Lambda.** Python 3.12 runtime. Role is the `CONSUMER_ROLE_NAME` ARN. Pass `CLUSTER_ID`, `DB_NAME`, and `DB_USER` as environment variables so the function knows which cluster to talk to. The function source is intentionally short and explicit so the cert-relevant details (Data API polling, MERGE invocation, log lines that Checkpoint 3 reads) are easy to find.
2. **Create the event-source mapping.** `lambda.create_event_source_mapping(EventSourceArn=stream_arn, FunctionName=CONSUMER_LAMBDA_NAME, StartingPosition="LATEST", BatchSize=100, Enabled=True)`. `LATEST` means records produced BEFORE the mapping was created are not consumed; the producer in Task 3 must run AFTER this cell finishes. Store the returned UUID in `EVENT_MAPPING_UUID` and update the manifest entry so cleanup can delete the mapping.
3. **Wait for the mapping to reach `Enabled`.** The mapping starts in `Creating` and transitions to `Enabled` in 10 to 30 seconds. The Checkpoint 2 validator reads `State == "Enabled"` so the cell waits before returning.

Checkpoint 2 confirms the mapping exists for `(CONSUMER_LAMBDA_NAME, stream_arn)`, `BatchSize == 100`, `StartingPosition == "LATEST"`, `State == "Enabled"`, the consumer Lambda's `Runtime == "python3.12"`, and `Role == consumer_role_arn`.


In [ ]:
# NBVAL_SKIP
# Task 2: Package the consumer Lambda, deploy it, create the event-source
# mapping. Store the mapping UUID in EVENT_MAPPING_UUID and update the
# CLEANUP_MANIFEST entry so cleanup can delete the mapping.

# Consumer Lambda source. boto3.client inside this source string uses the
# Lambda execution role and does not need aws_session_token (exception
# noted; the notebook-level rule does require session token on every
# boto3.client call). Pattern: batch all records into a single INSERT,
# then CALL merge_order_events().
consumer_source = """
import base64
import json
import os
import time

import boto3

CLUSTER_ID = os.environ["CLUSTER_ID"]
DB_NAME = os.environ["DB_NAME"]
DB_USER = os.environ["DB_USER"]


def _wait(rd, sid, timeout_s=30):
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        desc = rd.describe_statement(Id=sid)
        if desc["Status"] in ("FINISHED", "FAILED", "ABORTED"):
            if desc["Status"] != "FINISHED":
                raise RuntimeError(f"{sid} ended {desc[\"Status\"]}: {desc.get(\"Error\")}")
            return desc
        time.sleep(0.2)
    raise RuntimeError(f"{sid} did not finish in {timeout_s}s")


def handler(event, context):
    # boto3.client below uses the Lambda execution role; no session token.
    rd = boto3.client("redshift-data")
    records = event.get("Records", [])
    rows = []
    for r in records:
        raw = base64.b64decode(r["kinesis"]["data"])
        payload = json.loads(raw)
        rows.append((
            int(payload["order_id"]),
            int(payload["customer_id"]),
            int(payload["amount_cents"]),
            int(payload["produced_at"]),
        ))
    record_count = len(rows)
    print(f"recordCount={record_count}")
    if record_count == 0:
        return {"recordCount": 0}

    values = ", ".join(
        f"({oid}, {cid}, {amt}, {pat})" for (oid, cid, amt, pat) in rows
    )
    insert_sql = (
        "INSERT INTO order_events_stage "
        "(order_id, customer_id, amount_cents, produced_at) VALUES "
        + values
        + ";"
    )
    ins = rd.execute_statement(
        ClusterIdentifier=CLUSTER_ID,
        Database=DB_NAME,
        DbUser=DB_USER,
        Sql=insert_sql,
    )
    _wait(rd, ins["Id"])

    proc = rd.execute_statement(
        ClusterIdentifier=CLUSTER_ID,
        Database=DB_NAME,
        DbUser=DB_USER,
        Sql="CALL merge_order_events();",
    )
    _wait(rd, proc["Id"])

    return {"recordCount": record_count}
"""

c_zbuf = io.BytesIO()
with zipfile.ZipFile(c_zbuf, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.writestr("index.py", consumer_source)
consumer_zip_bytes = c_zbuf.getvalue()

# IAM role propagation can take a few seconds; retry on
# InvalidParameterValueException up to 60 seconds.
_lambda_deadline = time.time() + 60
while True:
    try:
        # YOUR CODE: lambda_client.create_function(
        #   FunctionName=CONSUMER_LAMBDA_NAME,
        #   Runtime="python3.12",
        #   Role=consumer_role_arn,
        #   Handler="index.handler",
        #   Code={"ZipFile": consumer_zip_bytes},
        #   Timeout=60,
        #   Environment={"Variables": {
        #     "CLUSTER_ID": CLUSTER_ID,
        #     "DB_NAME": DB_NAME,
        #     "DB_USER": DB_USER,
        #   }},
        #   Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
        # ).
        break
    except ClientError as e:
        if e.response["Error"]["Code"] == "InvalidParameterValueException" and time.time() < _lambda_deadline:
            time.sleep(5)
            continue
        raise
consumer_lambda_arn = f"arn:aws:lambda:{REGION}:{ACCOUNT_ID}:function:{CONSUMER_LAMBDA_NAME}"
print(f"Consumer Lambda created: {consumer_lambda_arn}")

# Create the event-source mapping. The Kinesis trigger needs the function
# role to allow kinesis:GetRecords; the consumer role inline policy in
# Task 1 already grants this. BatchSize=100 matches the brief; LATEST means
# the producer in Task 3 must run AFTER this cell finishes.
# YOUR CODE: esm = lambda_client.create_event_source_mapping(
#   EventSourceArn=stream_arn,
#   FunctionName=CONSUMER_LAMBDA_NAME,
#   StartingPosition="LATEST",
#   BatchSize=100,
#   Enabled=True,
# ).
# EVENT_MAPPING_UUID = esm["UUID"]
print(f"Event-source mapping UUID: {EVENT_MAPPING_UUID}")

# Update the manifest entry placeholder set in cell 4 now that we know
# the mapping UUID.
for r in CLEANUP_MANIFEST:
    if r.type == "lambda_event_source_mapping":
        r.id = EVENT_MAPPING_UUID or "(no UUID; create_event_source_mapping did not run)"
        r.cli_delete_command = (
            f"aws lambda delete-event-source-mapping --uuid {r.id}"
        )
        break

# Wait for the mapping to reach Enabled.
mapping_deadline = time.time() + 120
while time.time() < mapping_deadline:
    detail = lambda_client.get_event_source_mapping(UUID=EVENT_MAPPING_UUID)
    state_ = detail.get("State", "")
    if state_ == "Enabled":
        break
    if state_ in ("Disabled", "Failed"):
        raise RuntimeError(f"Event-source mapping in terminal state {state_}: {detail}")
    time.sleep(5)
print(f"Event-source mapping {EVENT_MAPPING_UUID} is Enabled.")


In [ ]:
# NBVAL_SKIP
# Checkpoint 2: event-source mapping exists with BatchSize=100,
# StartingPosition=LATEST, State=Enabled; consumer Lambda Runtime=python3.12
# and Role matches consumer_role_arn.


def checkpoint_2(session):
    try:
        lam = boto3.client(
            "lambda",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        try:
            fn = lam.get_function(FunctionName=CONSUMER_LAMBDA_NAME)
        except ClientError as e:
            if e.response["Error"]["Code"] == "ResourceNotFoundException":
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Consumer Lambda {CONSUMER_LAMBDA_NAME!r} does not "
                        f"exist. Run the Task 2 cell to create it."
                    ),
                )
            return CheckpointResult(status="error", error_reason=str(e))
        cfg = fn.get("Configuration", {})
        runtime = cfg.get("Runtime", "")
        if runtime != "python3.12":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Consumer Lambda Runtime is {runtime!r}, expected "
                    f"'python3.12'. Recreate the function with "
                    f"Runtime='python3.12' or use lambda.update_function_configuration."
                ),
            )
        role_ = cfg.get("Role", "")
        expected_role = f"arn:aws:iam::{ACCOUNT_ID}:role/{CONSUMER_ROLE_NAME}"
        if role_ != expected_role:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Consumer Lambda Role is {role_!r}, expected "
                    f"{expected_role!r}. The Kinesis trigger needs the "
                    f"consumer role with kinesis + redshift-data + logs grants."
                ),
            )

        # Validate consumer-role inline policy covers the required action
        # set. Accept exact actions, scoped wildcards (kinesis:*,
        # redshift-data:*, logs:*), or the universal wildcard (*).
        iam_client = boto3.client(
            "iam",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        required_actions = {
            "kinesis:GetRecords",
            "kinesis:GetShardIterator",
            "kinesis:DescribeStream",
            "kinesis:ListShards",
            "redshift-data:ExecuteStatement",
            "redshift-data:DescribeStatement",
            "redshift-data:GetStatementResult",
            "logs:CreateLogGroup",
            "logs:CreateLogStream",
            "logs:PutLogEvents",
        }
        granted_actions: set[str] = set()
        wildcards: set[str] = set()
        try:
            pol_names = iam_client.list_role_policies(
                RoleName=CONSUMER_ROLE_NAME
            ).get("PolicyNames", [])
            for pn in pol_names:
                pol = iam_client.get_role_policy(
                    RoleName=CONSUMER_ROLE_NAME, PolicyName=pn
                )
                doc = pol.get("PolicyDocument", {})
                for stmt in doc.get("Statement", []):
                    if stmt.get("Effect") != "Allow":
                        continue
                    acts = stmt.get("Action", [])
                    if isinstance(acts, str):
                        acts = [acts]
                    for a in acts:
                        if a == "*":
                            wildcards.add("*")
                        elif a.endswith(":*"):
                            wildcards.add(a)
                        else:
                            granted_actions.add(a)
        except ClientError as e:
            return CheckpointResult(
                status="error",
                error_reason=(
                    f"Could not read inline policies on "
                    f"{CONSUMER_ROLE_NAME!r}: {e}"
                ),
            )

        def covered(action: str) -> bool:
            if "*" in wildcards:
                return True
            prefix = action.split(":", 1)[0]
            if f"{prefix}:*" in wildcards:
                return True
            return action in granted_actions

        missing = sorted(a for a in required_actions if not covered(a))
        if missing:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Consumer role inline policy is missing required actions: "
                    f"{missing}. The Lambda needs kinesis read, "
                    f"redshift-data Execute/Describe/GetResult, and the three "
                    f"logs actions; scoped wildcards (kinesis:*, "
                    f"redshift-data:*, logs:*) and * are accepted."
                ),
            )

        # Event-source mapping.
        mappings_resp = lam.list_event_source_mappings(
            FunctionName=CONSUMER_LAMBDA_NAME
        )
        target_arn = f"arn:aws:kinesis:{REGION}:{ACCOUNT_ID}:stream/{STREAM_NAME}"
        candidates = [
            m for m in mappings_resp.get("EventSourceMappings", [])
            if m.get("EventSourceArn") == target_arn
        ]
        if not candidates:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"No event-source mapping found for consumer Lambda "
                    f"{CONSUMER_LAMBDA_NAME!r} pointing at stream "
                    f"{STREAM_NAME!r}. Run the Task 2 cell to create it."
                ),
            )
        m = candidates[0]
        batch = m.get("BatchSize", 0)
        if batch != 100:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Event-source mapping BatchSize is {batch}, expected 100."
                ),
            )
        sp = m.get("StartingPosition", "")
        if sp != "LATEST":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Event-source mapping StartingPosition is {sp!r}, "
                    f"expected 'LATEST'."
                ),
            )
        state_ = m.get("State", "")
        if state_ != "Enabled":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Event-source mapping State is {state_!r}, expected "
                    f"'Enabled'. Wait 15 seconds and re-run; mappings "
                    f"transition from Creating to Enabled in 10 to 30 seconds."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(2, checkpoint_2)


<details><summary>Hint 1 (nudge)</summary>

The checkpoint walks the consumer Lambda config (`Runtime == "python3.12"`, `Role` matches `consumer_role_arn`), the consumer role inline policy (must cover the kinesis read actions, the redshift-data actions, and the logs actions; scoped wildcards like `kinesis:*` or `redshift-data:*` and the universal `*` are accepted), then the event-source mapping (`BatchSize == 100`, `StartingPosition == "LATEST"`, `State == "Enabled"`). Read the failure reason. The most common miss is forgetting the `Environment` block on `create_function`; the consumer crashes at runtime with a `KeyError` on `CLUSTER_ID` and the mapping flips to `Disabled` after 5 invocation failures, which kills Checkpoint 2 on `State`. The second most common miss is `Runtime` left at the boto3 default; pin `Runtime="python3.12"` explicitly.

</details>

<details><summary>Hint 2 (direction)</summary>

Two API calls in this task. `lambda_client.create_function(FunctionName=CONSUMER_LAMBDA_NAME, Runtime="python3.12", Role=consumer_role_arn, Handler="index.handler", Code={"ZipFile": consumer_zip_bytes}, Timeout=60, Environment={"Variables": {"CLUSTER_ID": CLUSTER_ID, "DB_NAME": DB_NAME, "DB_USER": DB_USER}}, Tags={LAB_TAG_KEY: LAB_TAG_VALUE})` inside the retry loop (IAM role propagation can take a few seconds). Then `esm = lambda_client.create_event_source_mapping(EventSourceArn=stream_arn, FunctionName=CONSUMER_LAMBDA_NAME, StartingPosition="LATEST", BatchSize=100, Enabled=True)` and `EVENT_MAPPING_UUID = esm["UUID"]`. The poll-until-Enabled loop is already wired below the call. If `create_function` returns `InvalidParameterValueException: The role defined for the function cannot be assumed by Lambda`, wait 10 seconds and retry; the role exists but propagation has not completed.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 2:

```python
# Inside the retry loop:
lambda_client.create_function(
    FunctionName=CONSUMER_LAMBDA_NAME,
    Runtime="python3.12",
    Role=consumer_role_arn,
    Handler="index.handler",
    Code={"ZipFile": consumer_zip_bytes},
    Timeout=60,
    Environment={"Variables": {
        "CLUSTER_ID": CLUSTER_ID,
        "DB_NAME": DB_NAME,
        "DB_USER": DB_USER,
    }},
    Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
)

esm = lambda_client.create_event_source_mapping(
    EventSourceArn=stream_arn,
    FunctionName=CONSUMER_LAMBDA_NAME,
    StartingPosition="LATEST",
    BatchSize=100,
    Enabled=True,
)
EVENT_MAPPING_UUID = esm["UUID"]
```

If `create_event_source_mapping` returns `InvalidParameterValueException: Cannot access stream`, the consumer role inline policy is missing the kinesis read actions on the stream ARN; re-run `iam.put_role_policy` with the policy from Task 1. If the mapping creates but `State` stays in `Creating` for more than 60 seconds, AWS is throttling the Kinesis poller; wait and re-check `get_event_source_mapping`. If `State` flips to `Disabled` after creation, the consumer crashed on its first 5 invocations; check CloudWatch Logs for `/aws/lambda/labrun-kinesis-streams-lambda-redshift-consumer` and look for an `Environment` `KeyError`.

</details>


**Wallet check.** Cluster meter still running at $0.25 per hour; Kinesis meter still running at $0.015 per shard-hour. The Lambda function itself is free at this volume (well inside the 1M-requests + 400k GB-s monthly free tier). The event-source mapping is free; AWS does not bill for the poller. The S3 bucket holds nothing of consequence. Total session cost so far: dominated by cluster wall-clock since `create_cluster` returned.


## Task 3: Produce a 500-record burst, wait 60 seconds, and confirm the Lambda invoked at least 5 times with non-zero record counts

The mapping is wired, `StartingPosition=LATEST`, so records you produce now will flow through. The pattern: emit 500 records as 5 batches of 100 via `kinesis.put_records`, then wait 60 seconds for the poller to deliver every batch and for the Lambda to finish `INSERT` plus `CALL merge_order_events()` on each.

Three pieces in this task:

1. **Build and put 5 batches of 100 records.** Each record is a JSON payload with `order_id`, `customer_id`, `amount_cents`, and `produced_at` (the epoch-millisecond timestamp written immediately before `put_records`, so Checkpoint 4 can compute end-to-end latency). `PartitionKey` is `str(order_id)` so the 1-shard stream load-balances reasonably.
2. **Wait 60 seconds.** Then check the staging table count and the fact-table count via the Redshift Data API. The staging table is truncated by the stored procedure after each MERGE; the count there reflects only the most recent batch in flight. The fact table is the durable target.
3. **Run a CloudWatch Logs Insights query** for the consumer's log group, counting the `recordCount=` log lines emitted by the handler. At least 5 invocations with `recordCount >= 100` proves the BatchSize=100 trigger fired 5 times.

Checkpoint 3 confirms the producer emitted 500 records, the CloudWatch Logs Insights query reports at least 5 invocations with `recordCount >= 100`, and the fact table holds at least 500 rows after the 60-second wait.


In [ ]:
# NBVAL_SKIP
# Task 3: Produce 500 records as 5 batches of 100, wait 60 seconds, then
# query CloudWatch Logs Insights for invocation count and Redshift for
# the staging + fact-table counts.

kinesis = boto3.client(
    "kinesis",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
logs = boto3.client(
    "logs",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
redshift_data = boto3.client(
    "redshift-data",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)


def run_redshift_sql(sql: str, with_result: bool = False) -> dict:
    start = redshift_data.execute_statement(
        ClusterIdentifier=CLUSTER_ID,
        Database=DB_NAME,
        DbUser=DB_USER,
        Sql=sql,
    )
    sid = start["Id"]
    deadline = time.time() + 60
    while time.time() < deadline:
        desc = redshift_data.describe_statement(Id=sid)
        status_ = desc["Status"]
        if status_ in ("FINISHED", "FAILED", "ABORTED"):
            if status_ != "FINISHED":
                raise RuntimeError(
                    f"redshift-data {sid} ended {status_}: {desc.get('Error', '(no error)')}"
                )
            if with_result and desc.get("HasResultSet"):
                desc["result"] = redshift_data.get_statement_result(Id=sid)
            return desc
        time.sleep(0.5)
    raise RuntimeError(f"redshift-data {sid} did not finish within 60s.")


# Deterministic burst of 5 batches x 100 records.
rng = random.Random(SEED)
PRODUCER_START_MS = int(time.time() * 1000)
print(f"Producer start (epoch ms): {PRODUCER_START_MS}")

next_order_id = 1
# YOUR CODE: For each batch in range(BURST_BATCHES), build a list of
# BURST_RECORDS_PER_BATCH records of the shape:
#   {"Data": json.dumps({
#       "order_id": <int>,
#       "customer_id": <int 1..500>,
#       "amount_cents": <int 100..99999>,
#       "produced_at": <int epoch ms recorded NOW>,
#   }).encode(),
#   "PartitionKey": str(<order_id>)}
# and call kinesis.put_records(StreamName=STREAM_NAME, Records=batch).
# Increment next_order_id by BURST_RECORDS_PER_BATCH each loop.
#
# Example:
# for batch_ix in range(BURST_BATCHES):
#     batch = []
#     for _ in range(BURST_RECORDS_PER_BATCH):
#         payload = {
#             "order_id": next_order_id,
#             "customer_id": rng.randint(1, 500),
#             "amount_cents": rng.randint(100, 99999),
#             "produced_at": int(time.time() * 1000),
#         }
#         batch.append({
#             "Data": json.dumps(payload).encode(),
#             "PartitionKey": str(next_order_id),
#         })
#         next_order_id += 1
#     resp = kinesis.put_records(StreamName=STREAM_NAME, Records=batch)
#     print(f"batch {batch_ix}: {resp['FailedRecordCount']} failed")

print(f"Produced {BURST_TOTAL_RECORDS} records across {BURST_BATCHES} batches.")

# Wait for the poller plus the consumer plus the Data API plus the
# stored procedure to drain. 60 seconds is the brief's required wait.
print("Waiting 60 seconds for Kinesis poller + Lambda + Redshift Data API...")
time.sleep(60)

# Verify the fact table has all 500 rows.
fact_desc = run_redshift_sql("SELECT count(*) FROM orders_fact;", with_result=True)
fact_records = fact_desc.get("result", {}).get("Records", [])
fact_count = int(fact_records[0][0]["longValue"]) if fact_records else 0
print(f"orders_fact row count: {fact_count}")

# CloudWatch Logs Insights: count invocations with recordCount >= 100.
log_group = f"/aws/lambda/{CONSUMER_LAMBDA_NAME}"

start_ts = int(PRODUCER_START_MS / 1000) - 60
end_ts = int(time.time()) + 1
query_str = (
    "fields @timestamp, @message "
    "| filter @message like /recordCount=/ "
    "| parse @message 'recordCount=*' as rc "
    "| filter rc >= 100 "
    "| stats count() as invocations"
)
qres = logs.start_query(
    logGroupName=log_group,
    startTime=start_ts,
    endTime=end_ts,
    queryString=query_str,
)
qid = qres["queryId"]
insights_deadline = time.time() + 60
invocations = 0
while time.time() < insights_deadline:
    qstat = logs.get_query_results(queryId=qid)
    if qstat.get("status") in ("Complete", "Failed", "Cancelled"):
        if qstat.get("status") == "Complete":
            for row in qstat.get("results", []):
                for field in row:
                    if field.get("field") == "invocations":
                        invocations = int(field.get("value", "0"))
        break
    time.sleep(2)
print(f"CloudWatch Logs Insights invocations with recordCount >= 100: {invocations}")


In [ ]:
# NBVAL_SKIP
# Checkpoint 3: producer emitted 500 records, CloudWatch Logs Insights
# reports >= 5 invocations with recordCount >= 100, fact table has >= 500
# rows after the 60-second wait.


def checkpoint_3(session):
    try:
        rd = boto3.client(
            "redshift-data",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        cw = boto3.client(
            "logs",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        def run_sql(sql: str) -> list:
            start = rd.execute_statement(
                ClusterIdentifier=CLUSTER_ID,
                Database=DB_NAME,
                DbUser=DB_USER,
                Sql=sql,
            )
            sid = start["Id"]
            deadline = time.time() + 60
            while time.time() < deadline:
                desc = rd.describe_statement(Id=sid)
                status_ = desc["Status"]
                if status_ in ("FINISHED", "FAILED", "ABORTED"):
                    if status_ != "FINISHED":
                        raise RuntimeError(
                            f"{sid} ended {status_}: {desc.get('Error', '(no error)')}"
                        )
                    if desc.get("HasResultSet"):
                        res = rd.get_statement_result(Id=sid)
                        return res.get("Records", [])
                    return []
                time.sleep(0.5)
            raise RuntimeError(f"{sid} did not finish within 60s.")

        # Fact-table count.
        try:
            records = run_sql("SELECT count(*) FROM orders_fact;")
        except RuntimeError as e:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Cannot SELECT from orders_fact: {e}. Confirm Task 1 "
                    f"created the table and the consumer Lambda is wired."
                ),
            )
        fact_count = int(records[0][0]["longValue"]) if records else 0
        if fact_count < BURST_TOTAL_RECORDS:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"orders_fact has {fact_count} rows, expected at least "
                    f"{BURST_TOTAL_RECORDS}. The producer may not have run, "
                    f"or the consumer Lambda failed mid-batch. Check "
                    f"CloudWatch Logs for /aws/lambda/{CONSUMER_LAMBDA_NAME} "
                    f"for errors. If the staging table has rows but "
                    f"orders_fact does not, the stored procedure did not run; "
                    f"call merge_order_events() manually."
                ),
            )

        # CloudWatch Logs Insights: at least 5 invocations with recordCount >= 100.
        log_group = f"/aws/lambda/{CONSUMER_LAMBDA_NAME}"
        try:
            cw.describe_log_groups(logGroupNamePrefix=log_group)
        except ClientError as e:
            return CheckpointResult(
                status="error",
                error_reason=f"describe_log_groups failed: {e}",
            )
        start_ts = int(time.time()) - 1800  # last 30 minutes
        end_ts = int(time.time()) + 1
        query_str = (
            "fields @timestamp, @message "
            "| filter @message like /recordCount=/ "
            "| parse @message 'recordCount=*' as rc "
            "| filter rc >= 100 "
            "| stats count() as invocations"
        )
        qres = cw.start_query(
            logGroupName=log_group,
            startTime=start_ts,
            endTime=end_ts,
            queryString=query_str,
        )
        qid = qres["queryId"]
        deadline = time.time() + 60
        invocations = 0
        while time.time() < deadline:
            qstat = cw.get_query_results(queryId=qid)
            if qstat.get("status") in ("Complete", "Failed", "Cancelled"):
                if qstat.get("status") == "Complete":
                    for row in qstat.get("results", []):
                        for field in row:
                            if field.get("field") == "invocations":
                                invocations = int(field.get("value", "0"))
                break
            time.sleep(2)
        if invocations < BURST_BATCHES:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"CloudWatch Logs Insights reports {invocations} "
                    f"invocations with recordCount >= 100, expected at least "
                    f"{BURST_BATCHES}. The producer may have produced fewer "
                    f"than {BURST_TOTAL_RECORDS} records, the BatchSize may "
                    f"not be 100 on the mapping, or the consumer log line "
                    f"may not match the 'recordCount=' format. Check the "
                    f"consumer source string for the print statement and "
                    f"re-run Task 3 if the format changed."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(3, checkpoint_3)


<details><summary>Hint 1 (nudge)</summary>

The checkpoint walks two things: `SELECT count(*) FROM orders_fact` returns at least 500 (the producer emitted 500 records and the stored procedure copied them all into the fact table) and the CloudWatch Logs Insights query against `/aws/lambda/labrun-kinesis-streams-lambda-redshift-consumer` reports at least 5 invocations with `recordCount >= 100`. Read the failure reason. The most common miss is forgetting to call `kinesis.put_records` inside the loop (the cell builds the batch but never sends it). The second most common miss is `produced_at` being passed as a Python float instead of an `int` of epoch milliseconds; the consumer's `INSERT` cast then fails and the fact table stays empty.

</details>

<details><summary>Hint 2 (direction)</summary>

One loop. For each `batch_ix` in `range(BURST_BATCHES)`, build a list of `BURST_RECORDS_PER_BATCH` records. Each record is `{"Data": json.dumps(payload).encode(), "PartitionKey": str(order_id)}` where `payload` is `{"order_id": ..., "customer_id": rng.randint(1, 500), "amount_cents": rng.randint(100, 99999), "produced_at": int(time.time() * 1000)}`. Call `kinesis.put_records(StreamName=STREAM_NAME, Records=batch)` and print `resp["FailedRecordCount"]` so you spot any partial failures. Increment `next_order_id` by `BURST_RECORDS_PER_BATCH` per loop so order IDs stay unique. The 60-second wait below the loop is fixed; it gives the poller plus the consumer plus the Redshift Data API plus the stored procedure time to drain.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 3:

```python
for batch_ix in range(BURST_BATCHES):
    batch = []
    for _ in range(BURST_RECORDS_PER_BATCH):
        payload = {
            "order_id": next_order_id,
            "customer_id": rng.randint(1, 500),
            "amount_cents": rng.randint(100, 99999),
            "produced_at": int(time.time() * 1000),
        }
        batch.append({
            "Data": json.dumps(payload).encode(),
            "PartitionKey": str(next_order_id),
        })
        next_order_id += 1
    resp = kinesis.put_records(StreamName=STREAM_NAME, Records=batch)
    print(f"batch {batch_ix}: failed={resp['FailedRecordCount']}")
```

If `kinesis.put_records` returns `ProvisionedThroughputExceededException`, the 1-shard stream is being asked for more than 1000 records per second; the 5-batch cadence with `time.sleep(0.2)` between batches (add it if needed) keeps you under. If `FailedRecordCount > 0`, inspect `resp["Records"]` for `ErrorCode`; the most common is `InternalFailure` on a transient AWS hiccup, retry the failed records. If `orders_fact` count is zero after the 60-second wait but the Insights query reports invocations, the consumer is dying inside the `INSERT` or `CALL`; check `/aws/lambda/labrun-kinesis-streams-lambda-redshift-consumer` for the actual error. If the staging table has rows and the fact table is empty, the `CALL merge_order_events()` step never ran; call it once manually with `redshift-data.execute_statement(... Sql='CALL merge_order_events();')` and confirm `orders_fact` fills.

</details>


**Wallet check.** Cluster meter still running at $0.25 per hour; Kinesis meter still running at $0.015 per shard-hour. The 500 PUT payload units for the burst are pennies-per-million (negligible at this volume). Each Lambda invocation is on the order of half a second of 128 MB compute, well inside the always-free 400k GB-seconds monthly tier. CloudWatch Logs ingestion for the consumer prints is a few KB; well inside the free 5 GB/month. Total session cost so far: still dominated by cluster wall-clock; you are at $0.10 to $0.20.


## Task 4: Prove end-to-end latency is under 45 seconds

The producer records carry a `produced_at` epoch-millisecond field. The consumer Lambda copies that field into `order_events_stage` via the `INSERT`, the stored procedure copies it into `orders_fact` via the MERGE. So the latency from `put_records` to fact-table visibility is `now - max(produced_at)` in milliseconds, read directly from Redshift.

Three pieces in this task:

1. **Query `SELECT max(produced_at) FROM orders_fact;`** via the Redshift Data API. The value is the most recent `produced_at` epoch-ms timestamp the consumer saw.
2. **Capture `now`** as `int(time.time() * 1000)` at the same Python instant you read the result.
3. **Assert `now - max_produced_at < 45_000`** AND `orders_fact` row count is exactly `BURST_TOTAL_RECORDS` (500). The first assertion proves the pipeline meets SLA; the second proves no records were dropped end to end.

Checkpoint 4 confirms both assertions. If the latency assertion fails, the producer may have run before the event-source mapping reached `Enabled` (records from before mapping creation are dropped under `StartingPosition=LATEST`); re-run Task 3.


In [ ]:
# NBVAL_SKIP
# Task 4: Query max(produced_at) from orders_fact, compute now - that
# value, and store the result in LATENCY_MS for Checkpoint 4 to read.

redshift_data = boto3.client(
    "redshift-data",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)


def run_redshift_sql(sql: str, with_result: bool = False) -> dict:
    start = redshift_data.execute_statement(
        ClusterIdentifier=CLUSTER_ID,
        Database=DB_NAME,
        DbUser=DB_USER,
        Sql=sql,
    )
    sid = start["Id"]
    deadline = time.time() + 60
    while time.time() < deadline:
        desc = redshift_data.describe_statement(Id=sid)
        status_ = desc["Status"]
        if status_ in ("FINISHED", "FAILED", "ABORTED"):
            if status_ != "FINISHED":
                raise RuntimeError(
                    f"redshift-data {sid} ended {status_}: {desc.get('Error', '(no error)')}"
                )
            if with_result and desc.get("HasResultSet"):
                desc["result"] = redshift_data.get_statement_result(Id=sid)
            return desc
        time.sleep(0.5)
    raise RuntimeError(f"redshift-data {sid} did not finish within 60s.")


# YOUR CODE: Replace the two placeholders below.
# Query 1: SELECT max(produced_at) FROM orders_fact;
# Query 2: SELECT count(*) FROM orders_fact;
# Then compute now_ms - max_produced_at and store both in module-level
# variables.
#
# Example:
# max_desc = run_redshift_sql(
#     "SELECT max(produced_at) FROM orders_fact;", with_result=True
# )
# max_records = max_desc.get("result", {}).get("Records", [])
# max_produced_at = int(max_records[0][0]["longValue"]) if max_records else 0
#
# count_desc = run_redshift_sql(
#     "SELECT count(*) FROM orders_fact;", with_result=True
# )
# count_records = count_desc.get("result", {}).get("Records", [])
# fact_count = int(count_records[0][0]["longValue"]) if count_records else 0

max_produced_at = 0
fact_count = 0

now_ms = int(time.time() * 1000)
LATENCY_MS = now_ms - max_produced_at
FACT_ROW_COUNT = fact_count

print(f"max(produced_at): {max_produced_at}")
print(f"now (epoch ms): {now_ms}")
print(f"end-to-end latency: {LATENCY_MS} ms (SLA: under {LATENCY_SLA_MS} ms)")
print(f"orders_fact row count: {FACT_ROW_COUNT}")


In [ ]:
# NBVAL_SKIP
# Checkpoint 4: end-to-end latency under 45 seconds AND orders_fact has
# exactly BURST_TOTAL_RECORDS rows. Reads LATENCY_MS and FACT_ROW_COUNT
# populated by Task 4. If either is missing or stale, the checkpoint
# re-queries Redshift itself.


def checkpoint_4(session):
    global LATENCY_MS, FACT_ROW_COUNT
    try:
        rd = boto3.client(
            "redshift-data",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        def run_sql(sql: str) -> list:
            start = rd.execute_statement(
                ClusterIdentifier=CLUSTER_ID,
                Database=DB_NAME,
                DbUser=DB_USER,
                Sql=sql,
            )
            sid = start["Id"]
            deadline = time.time() + 60
            while time.time() < deadline:
                desc = rd.describe_statement(Id=sid)
                status_ = desc["Status"]
                if status_ in ("FINISHED", "FAILED", "ABORTED"):
                    if status_ != "FINISHED":
                        raise RuntimeError(
                            f"{sid} ended {status_}: {desc.get('Error', '(no error)')}"
                        )
                    if desc.get("HasResultSet"):
                        res = rd.get_statement_result(Id=sid)
                        return res.get("Records", [])
                    return []
                time.sleep(0.5)
            raise RuntimeError(f"{sid} did not finish within 60s.")

        latency = globals().get("LATENCY_MS", 0)
        fact_count = globals().get("FACT_ROW_COUNT", 0)
        if latency <= 0 or fact_count <= 0:
            # Recompute from scratch if Task 4 did not populate the globals.
            max_records = run_sql("SELECT max(produced_at) FROM orders_fact;")
            max_produced_at = (
                int(max_records[0][0]["longValue"]) if max_records else 0
            )
            count_records = run_sql("SELECT count(*) FROM orders_fact;")
            fact_count = (
                int(count_records[0][0]["longValue"]) if count_records else 0
            )
            now_ms = int(time.time() * 1000)
            latency = now_ms - max_produced_at
            LATENCY_MS = latency
            FACT_ROW_COUNT = fact_count

        if fact_count != BURST_TOTAL_RECORDS:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"orders_fact row count is {fact_count}, expected exactly "
                    f"{BURST_TOTAL_RECORDS}. If less, the consumer dropped "
                    f"records (check CloudWatch Logs). If more, the producer "
                    f"ran twice; truncate orders_fact and re-run Task 3."
                ),
            )

        if latency >= LATENCY_SLA_MS:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"End-to-end latency is {latency} ms; SLA is under "
                    f"{LATENCY_SLA_MS} ms. The most common cause is the "
                    f"producer running before the event-source mapping "
                    f"reached Enabled (StartingPosition=LATEST drops earlier "
                    f"records). Re-run Task 3 with the mapping confirmed "
                    f"Enabled, then re-run this cell."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(4, checkpoint_4)


<details><summary>Hint 1 (nudge)</summary>

The checkpoint walks two assertions: `orders_fact` row count equals exactly 500, and `now_ms - max(produced_at) < 45000`. Read the failure reason. The most common miss is reading the row count via `intValue` instead of `longValue`; Redshift's `count(*)` returns a BIGINT and the Data API result field key is `longValue`. The second most common miss is computing latency in seconds (oops, `time.time()`) instead of milliseconds; multiply by 1000.

</details>

<details><summary>Hint 2 (direction)</summary>

Two `run_redshift_sql` calls and two integer reads. `run_redshift_sql("SELECT max(produced_at) FROM orders_fact;", with_result=True)` returns a describe payload whose `result.Records[0][0]["longValue"]` is the most recent `produced_at` epoch ms. `run_redshift_sql("SELECT count(*) FROM orders_fact;", with_result=True)` returns the row count the same way. Then `now_ms = int(time.time() * 1000)` and `LATENCY_MS = now_ms - max_produced_at`. Print all three values so you can confirm the math is right before the checkpoint asserts.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 4:

```python
max_desc = run_redshift_sql(
    "SELECT max(produced_at) FROM orders_fact;", with_result=True
)
max_records = max_desc.get("result", {}).get("Records", [])
max_produced_at = int(max_records[0][0]["longValue"]) if max_records else 0

count_desc = run_redshift_sql(
    "SELECT count(*) FROM orders_fact;", with_result=True
)
count_records = count_desc.get("result", {}).get("Records", [])
fact_count = int(count_records[0][0]["longValue"]) if count_records else 0
```

If `max(produced_at)` returns 0 or a very old timestamp, the consumer never wrote any records; check the consumer Lambda CloudWatch Logs and confirm the event-source mapping is `Enabled`. If the row count is greater than 500, the producer ran twice; `run_redshift_sql("TRUNCATE orders_fact;")` and re-run Task 3 (the staging table is already empty after each `CALL merge_order_events()`). If the latency is way over 45 seconds (say 90 seconds or more), the producer ran before the event-source mapping reached `Enabled`; the records from that earlier run were dropped, and the records you see in `orders_fact` are from a re-run. Truncate, re-run Task 3 after confirming the mapping is `Enabled`, then re-run this cell.

</details>


**Wallet check.** Cluster meter still running at $0.25 per hour; Kinesis meter still running at $0.015 per shard-hour. The two `SELECT` queries in this task are free against the running cluster. Total session cost is dominated by wall-clock since `create_cluster` returned. If you have been moving fast, you are at $0.15 to $0.25 right now; the cleanup cell below stops both meters the moment the cluster reaches `Deleting` state and the stream returns `ResourceNotFoundException`.


In [ ]:
# NBVAL_SKIP
# Cleanup. Tear down every resource in CLEANUP_MANIFEST in reverse-creation,
# critical-first order per RESOURCE-SAFETY-SPEC Rule 2.
#
# labrun-checks 0.3.0 lacks adapters for redshift_cluster, kinesis_stream,
# eventbridge_rule, lambda_function, lambda_event_source_mapping, and
# security_group. The _LabAdapter below wraps AwsCleanupAdapter and
# dispatches those six types inline so cleanup works end-to-end without a
# labrun-checks release. When the package promotes these adapters in a
# future release, the wrapper can be removed and run_cleanup called against
# the default adapter.
#
# The Kinesis adapter waits for ResourceNotFoundException on describe_stream
# before returning since shards take ~30s to actually go away after
# delete_stream returns. The Redshift adapter waits for ClusterNotFound for
# the same reason.

import sys
from labrun_checks.adapters.aws import AwsCleanupAdapter


class _LabAdapter:
    """AwsCleanupAdapter wrapper that adds Lab 08 inline resource types.

    Inlined types: redshift_cluster, kinesis_stream, eventbridge_rule,
    lambda_function, lambda_event_source_mapping, security_group.
    Everything else (s3_bucket, iam_role) delegates to the bundled
    AwsCleanupAdapter unchanged.
    """

    def __init__(self) -> None:
        self._aws = AwsCleanupAdapter()

    def _client(self, service: str, credentials: dict):
        return boto3.client(
            service,
            aws_access_key_id=credentials["aws_access_key_id"],
            aws_secret_access_key=credentials["aws_secret_access_key"],
            aws_session_token=credentials.get("aws_session_token"),
            region_name=credentials.get("region", REGION),
        )

    def delete_resource(self, credentials: dict, resource) -> None:
        if resource.type == "redshift_cluster":
            client = self._client("redshift", credentials)
            try:
                client.delete_cluster(
                    ClusterIdentifier=resource.id,
                    SkipFinalClusterSnapshot=True,
                )
            except ClientError as e:
                code_ = e.response["Error"]["Code"]
                if code_ in ("ClusterNotFound", "ClusterNotFoundFault"):
                    return
                if code_ == "InvalidClusterState":
                    pass
                else:
                    raise
            wait_deadline = time.time() + 600
            while time.time() < wait_deadline:
                try:
                    client.describe_clusters(ClusterIdentifier=resource.id)
                except ClientError as e:
                    if e.response["Error"]["Code"] in ("ClusterNotFound", "ClusterNotFoundFault"):
                        return
                    raise
                time.sleep(15)
            return

        if resource.type == "kinesis_stream":
            client = self._client("kinesis", credentials)
            try:
                client.delete_stream(
                    StreamName=resource.id,
                    EnforceConsumerDeletion=True,
                )
            except ClientError as e:
                if e.response["Error"]["Code"] != "ResourceNotFoundException":
                    raise
            # Wait for ResourceNotFound on describe_stream before returning;
            # shards take ~30s to actually disappear.
            wait_deadline = time.time() + 180
            while time.time() < wait_deadline:
                try:
                    client.describe_stream(StreamName=resource.id)
                except ClientError as e:
                    if e.response["Error"]["Code"] == "ResourceNotFoundException":
                        return
                    raise
                time.sleep(5)
            return

        if resource.type == "eventbridge_rule":
            client = self._client("events", credentials)
            try:
                client.remove_targets(
                    Rule=resource.id,
                    Ids=["1"],
                    Force=True,
                )
            except ClientError as e:
                if e.response["Error"]["Code"] not in (
                    "ResourceNotFoundException", "InternalException"
                ):
                    raise
            try:
                client.delete_rule(Name=resource.id)
            except ClientError as e:
                if e.response["Error"]["Code"] != "ResourceNotFoundException":
                    raise
            return

        if resource.type == "lambda_function":
            client = self._client("lambda", credentials)
            try:
                client.delete_function(FunctionName=resource.id)
            except ClientError as e:
                if e.response["Error"]["Code"] != "ResourceNotFoundException":
                    raise
            return

        if resource.type == "lambda_event_source_mapping":
            client = self._client("lambda", credentials)
            try:
                client.delete_event_source_mapping(UUID=resource.id)
            except ClientError as e:
                if e.response["Error"]["Code"] != "ResourceNotFoundException":
                    raise
            return

        if resource.type == "security_group":
            client = self._client("ec2", credentials)
            try:
                resp = client.describe_security_groups(
                    Filters=[{"Name": "group-name", "Values": [resource.id]}],
                )
                for sg in resp.get("SecurityGroups", []):
                    try:
                        client.delete_security_group(GroupId=sg["GroupId"])
                    except ClientError as e:
                        if e.response["Error"]["Code"] != "InvalidGroup.NotFound":
                            raise
            except ClientError as e:
                if e.response["Error"]["Code"] != "InvalidGroup.NotFound":
                    raise
            return

        return self._aws.delete_resource(credentials, resource)

    def describe_resource(self, credentials: dict, resource) -> bool:
        if resource.type == "redshift_cluster":
            client = self._client("redshift", credentials)
            try:
                client.describe_clusters(ClusterIdentifier=resource.id)
                return True
            except ClientError as e:
                if e.response["Error"]["Code"] in ("ClusterNotFound", "ClusterNotFoundFault"):
                    return False
                raise

        if resource.type == "kinesis_stream":
            client = self._client("kinesis", credentials)
            try:
                client.describe_stream(StreamName=resource.id)
                return True
            except ClientError as e:
                if e.response["Error"]["Code"] == "ResourceNotFoundException":
                    return False
                raise

        if resource.type == "eventbridge_rule":
            client = self._client("events", credentials)
            try:
                client.describe_rule(Name=resource.id)
                return True
            except ClientError as e:
                if e.response["Error"]["Code"] == "ResourceNotFoundException":
                    return False
                raise

        if resource.type == "lambda_function":
            client = self._client("lambda", credentials)
            try:
                client.get_function(FunctionName=resource.id)
                return True
            except ClientError as e:
                if e.response["Error"]["Code"] == "ResourceNotFoundException":
                    return False
                raise

        if resource.type == "lambda_event_source_mapping":
            client = self._client("lambda", credentials)
            try:
                client.get_event_source_mapping(UUID=resource.id)
                return True
            except ClientError as e:
                if e.response["Error"]["Code"] == "ResourceNotFoundException":
                    return False
                raise

        if resource.type == "security_group":
            client = self._client("ec2", credentials)
            try:
                resp = client.describe_security_groups(
                    Filters=[{"Name": "group-name", "Values": [resource.id]}],
                )
                return bool(resp.get("SecurityGroups", []))
            except ClientError as e:
                if e.response["Error"]["Code"] == "InvalidGroup.NotFound":
                    return False
                raise

        return self._aws.describe_resource(credentials, resource)

    def tag_scan(self, credentials: dict, lab_slug: str, region: str) -> list[str]:
        return self._aws.tag_scan(credentials, lab_slug, region)


# Empty the S3 bucket before run_cleanup tears it down. S3 buckets cannot
# be deleted while they contain objects.
print("Emptying the S3 bucket before teardown...")
s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
try:
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=BUCKET_NAME):
        keys = [{"Key": obj["Key"]} for obj in page.get("Contents", [])]
        if keys:
            s3.delete_objects(Bucket=BUCKET_NAME, Delete={"Objects": keys})
except ClientError as e:
    if e.response["Error"]["Code"] != "NoSuchBucket":
        print(f"Bucket emptying ran into: {e}. Continuing to cleanup.")

# Drop inline policies on the three roles before role-delete so the
# iam_role adapter does not block on attached policies.
iam_client = boto3.client(
    "iam",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
for role_name in (CONSUMER_ROLE_NAME, COPY_ROLE_NAME, SAFETY_NET_ROLE_NAME):
    try:
        listed = iam_client.list_role_policies(RoleName=role_name)
        for policy_name in listed.get("PolicyNames", []):
            iam_client.delete_role_policy(
                RoleName=role_name, PolicyName=policy_name
            )
    except ClientError as e:
        if e.response["Error"]["Code"] != "NoSuchEntity":
            print(f"Inline policy detach for {role_name} ran into: {e}. Continuing.")

result = run_cleanup(CLEANUP_MANIFEST, adapter=_LabAdapter())

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids: set[str] = set()
for w in result.warnings:
    for r in CLEANUP_MANIFEST:
        if r.id in w:
            failed_ids.add(r.id)
            break

critical_total = sum(1 for r in CLEANUP_MANIFEST if getattr(r, "critical", False))
standard_total = len(CLEANUP_MANIFEST) - critical_total
critical_destroyed = critical_total - sum(
    1 for r in CLEANUP_MANIFEST if getattr(r, "critical", False) and r.id in failed_ids
)
standard_destroyed = standard_total - sum(
    1 for r in CLEANUP_MANIFEST if not getattr(r, "critical", False) and r.id in failed_ids
)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your AWS account may still incur charges. Resolve before closing this notebook.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)


**Session total: $0.20 to $0.60.** Two meters this time, but the math is similar to Lab 06. Redshift `dc2.large` at $0.25 per hour is the dominant line item; the Kinesis 1-shard stream at $0.015 per shard-hour adds a few cents over a 45-to-60-minute session. Happy path with cleanup running clean: $0.20 to $0.30. Worst case (kernel died, atexit ran late, the 2-hour EventBridge safety net fired): $0.60. The catastrophic case (cleanup skipped, atexit failed, safety-net Lambda failed) would require three independent failures and is the reason the $20-per-month AWS billing alert exists. The cleanup cell stops both meters; the cluster goes to `Deleting` state and the stream returns `ResourceNotFoundException` the moment delete returns. On the cert-scenario shape (continuous order stream, multiple shards, multi-node `dc2.large` cluster), the same pattern runs at production cost; the optimization is to migrate to `ra3` with managed storage and Kinesis on-demand mode once the throughput pattern is known.


## These are not graded. They are for you.

Three questions worth sitting with before the exam.

1. You picked `BatchSize=100` on the event-source mapping. Walk the trade-off. What happens to end-to-end latency when `BatchSize` is 1 (record-by-record invocation, lowest latency, highest Lambda cost)? What happens when `BatchSize` is 10000 (max batch, lowest cost per record, latency-bounded by `MaximumBatchingWindowInSeconds`)? For Kinesis triggers, the poll interval is fixed (~200 ms); for SQS triggers, the poll model is different. The cert wants you to reason about latency, cost, and throughput jointly: a sub-second SLA pushes batch size low; a 5-minute SLA pushes batch size high. Identify two scenarios where you would set `BatchSize=1` and two where you would set `BatchSize=10000`.

2. The stored procedure runs the MERGE inside the Lambda invocation, on every batch. Walk through the alternatives. A scheduled MERGE (EventBridge fires every 30 seconds against the cluster) decouples ingestion from MERGE cost but adds latency. A Redshift Streaming Ingestion materialized view (Kinesis source plus auto-refresh) skips Lambda entirely and ingests directly into Redshift; this is the cert-canonical "fewest moving parts" pattern for streaming-into-Redshift. The lab uses the Lambda-plus-Data-API path because it shows the Data API semantics; production teams on Redshift would usually choose Streaming Ingestion. Identify when each pattern is the right tool.

3. The 2-hour EventBridge plus Lambda safety net cost about $0 to run and stopped a forgotten cluster plus stream from costing $44 over a week ($42 cluster + $2.52 stream). Walk through the alternatives. AWS Budgets with a forecasted-spend action can shut down resources, but only at the account level (too blunt for a lab). A CloudWatch alarm on `BillingEstimatedCharges` notifies but does not delete. A second EventBridge rule firing every 30 minutes could check the cluster's `ClusterCreateTime` plus the stream's `StreamCreationTimestamp` and delete after 2 hours. For each, identify when it is the right tool and when the one-shot safety net is the cleaner choice.

## Exam-Style Practice

**Q1.** A data engineer wires a Kinesis Data Streams source to a Lambda function via event-source mapping with `BatchSize=100` and `StartingPosition=LATEST`. After deploying, the team produces 1000 records via `kinesis.put_records` and observes that the Lambda processes only 600 of them. What is the most likely cause?

A) The Lambda function timed out before processing the full batch; increase the function timeout.

B) Some records were produced before the event-source mapping reached `State=Enabled`; with `StartingPosition=LATEST`, those records were not consumed.

C) The stream has fewer than 10 shards, so the Lambda concurrency cap dropped 400 records.

D) `BatchSize=100` means the Lambda sees only 100 records total; the other 900 are discarded.

<details><summary>Show answer</summary>

**B**.

- A) Wrong. Lambda timeout would surface as failed invocations in CloudWatch Logs, not silently dropped records; the mapping retries on function timeout by default.
- B) Right. `StartingPosition=LATEST` means the consumer starts at the tip of the stream at the moment the mapping reaches `Enabled`. Any records produced before that wall-clock instant are not consumed. The fix on production code is to wait for the mapping to reach `Enabled` (via `lambda.get_event_source_mapping` polling) before turning on the producer, or to use `StartingPosition=TRIM_HORIZON` to consume from the oldest record in the stream's 24-hour retention window. This lab waits explicitly in Task 2 for exactly this reason.
- C) Wrong. The Kinesis trigger does not have a per-shard concurrency cap that drops records; it has a per-shard parallelization factor (default 1) that limits parallelism, not delivery.
- D) Wrong. `BatchSize` is the max number of records per invocation, not a total cap. With `BatchSize=100`, processing 1000 records means 10 invocations of 100 records each.

</details>

**Q2.** A team must stream JSON order events into a Redshift fact table with under 60-second freshness. The platform team can choose between (a) a Kinesis Data Stream plus a Lambda consumer that uses the Redshift Data API to INSERT into a staging table and CALL a stored procedure that MERGEs into the fact table, or (b) Redshift Streaming Ingestion via a Kinesis source and an auto-refreshed materialized view. Which is the simpler architecture with fewer moving parts and lower operational overhead?

A) Option (a), because Lambda gives explicit control over batching and error handling.

B) Option (a), because the Data API path is cheaper than the materialized view refresh.

C) Option (b), because Redshift Streaming Ingestion connects the stream directly to a materialized view; no Lambda, no Data API, no stored procedure.

D) Option (b), because it does not require a Kinesis Data Stream.

<details><summary>Show answer</summary>

**C**.

- A) Wrong. Option (b) gives the same freshness with significantly fewer moving parts; the cert exam tests recognition of the "use the managed service" pattern over the "wire it yourself" pattern.
- B) Wrong. Materialized view auto-refresh on Streaming Ingestion is included in Redshift cluster costs; there is no additional per-row charge. Option (a) adds Lambda invocations plus Data API charges.
- C) Right. Redshift Streaming Ingestion (announced 2022; generally available 2023) lets you create an external schema on a Kinesis stream and a materialized view that auto-refreshes from the stream. The fact table is the materialized view; there is no Lambda, no Data API, no staging table, no stored procedure. The lab uses the Lambda-plus-Data-API path because it teaches the underlying primitives (Data API polling, batched INSERT, stored procedure), but a production team would choose Streaming Ingestion for fewer moving parts.
- D) Wrong. Option (b) still requires a Kinesis Data Stream; the materialized view consumes from it.

</details>

**Q3.** A Lambda function consuming from a Kinesis Data Stream via event-source mapping reports no invocations in the AWS Lambda console, even though `kinesis.put_records` confirms records are flowing into the stream. What is the most likely cause?

A) The Lambda function's execution role is missing `kinesis:GetRecords`, `kinesis:GetShardIterator`, `kinesis:DescribeStream`, and `kinesis:ListShards` on the stream ARN; the mapping silently fails the poll.

B) The Lambda function's `Runtime` is set to `python3.8`; Kinesis triggers only support runtimes from `python3.10` onward.

C) The Kinesis Data Stream is in PROVISIONED mode; Kinesis triggers only work with ON_DEMAND mode streams.

D) The Lambda function is in a different region than the Kinesis stream; cross-region triggers are not supported.

<details><summary>Show answer</summary>

**A**.

- A) Right. The event-source mapping uses the Lambda function's execution role to poll the stream. Without the kinesis read actions on the stream ARN (or scoped via `kinesis:*`, or via `*`), the mapping cannot fetch records; the poll fails silently from the user's perspective and the function is never invoked. The mapping's `LastProcessingResult` shows the actual error, and the mapping's `State` may flip to `Disabled` after sustained failures, but the symptom the user sees first is "no invocations."
- B) Wrong. Kinesis triggers support every supported Lambda runtime, including `python3.8` (deprecated but still supported for older functions) through `python3.12` and beyond.
- C) Wrong. Kinesis triggers work with both PROVISIONED and ON_DEMAND streams. ON_DEMAND is a newer billing mode; it does not change the trigger semantics.
- D) Wrong. Cross-region mappings are not a common pattern but the failure mode would be a `create_event_source_mapping` rejection at deploy time, not silent no-invocations.

</details>
